In [9]:
from typing import List, Dict, Any, Optional
import numpy as np
import torch
from torch_geometric.data import Data
from tqdm import tqdm


def build_edge_attr_from_dict(edge_feature, edge_index: torch.Tensor, default_dim: int = None) -> Optional[torch.Tensor]:
    """Align a dict {(u,v): feat} to edge_index -> [E, F]."""
    if edge_feature is None:
        return None
    if not isinstance(edge_feature, dict):
        # Fallback: accept arrays too
        try:
            arr = np.asarray(edge_feature)
            if arr.ndim == 1:
                arr = arr[:, None]
            E = edge_index.shape[1]
            if arr.shape[0] != E:
                if arr.ndim == 2 and arr.shape[1] == E:
                    arr = arr.T
                else:
                    print(f"[WARN] edge_feature array shape {arr.shape} != num_edges {E}; dropping edge_attr.")
                    return None
            return torch.tensor(arr, dtype=torch.float32)
        except Exception:
            print("[WARN] edge_feature not dict/array; dropping edge_attr.")
            return None

    # infer F from first value
    F = None
    for v in edge_feature.values():
        if isinstance(v, (list, tuple, np.ndarray)):
            F = int(np.asarray(v, dtype=float).size)
        else:
            F = 1
        break
    if F is None:
        return None
    if default_dim is not None:
        F = default_dim

    # normalize keys to int pairs and values to fixed length F
    def as_int_pair(k):
        try:
            u, v = k
            return int(u), int(v)
        except Exception:
            try:
                # keys like "(0, 6)"
                u, v = eval(k)
                return int(u), int(v)
            except Exception:
                return None

    norm = {}
    for k, v in edge_feature.items():
        kk = as_int_pair(k)
        if kk is None:
            continue
        if isinstance(v, (list, tuple, np.ndarray)):
            vv = np.asarray(v, dtype=float).reshape(-1)
        else:
            vv = np.array([float(v)], dtype=float)
        if vv.size != F:
            vv = vv[:F] if vv.size > F else np.pad(vv, (0, F - vv.size))
        norm[kk] = vv

    E = edge_index.shape[1]
    out = np.zeros((E, F), dtype=float)
    src = edge_index[0].tolist()
    dst = edge_index[1].tolist()
    for e, (u, v) in enumerate(zip(src, dst)):
        out[e] = norm.get((u, v), 0.0)
    return torch.tensor(out, dtype=torch.float32)


def to_edge_index_from_adj(adj: np.ndarray) -> torch.Tensor:
    idx = np.argwhere(adj != 0)
    if idx.size == 0:
        return torch.zeros((2, 0), dtype=torch.long)
    return torch.from_numpy(idx.T).long()


def build_lightcone_masks(shortest_path, measured_node_idx, N: int, M: int) -> torch.Tensor:
    if shortest_path is None or measured_node_idx is None:
        return torch.ones((N, M), dtype=torch.bool)
    sp = np.asarray(shortest_path)
    meas = np.asarray(measured_node_idx).astype(int)
    masks = np.zeros((N, M), dtype=bool)
    for m, node_id in enumerate(meas):
        if 0 <= node_id < N:
            col = sp[:, node_id]
            finite = np.isfinite(col) if np.issubdtype(col.dtype, np.floating) else (col >= 0)
            masks[:, m] = finite
    return torch.from_numpy(masks)


def convert_records_to_graphs(records: List[Dict[str, Any]], M_DEFAULT: Optional[int] = None) -> List[Data]:
    graphs = []
    for fp, rec in tqdm(records):
        # --- required-ish ---
        x = rec.get("x", None)
        adj = rec.get("adj", None)
        if x is None or adj is None:
            print(f"[WARN] {fp}: missing 'x' or 'adj'; skipping.")
            continue

        x = np.asarray(x, dtype=float)
        N = x.shape[0]

        # optional: concat positional encodings
        pe_feat = rec.get("pe_feat", None)
        if pe_feat is not None:
            pe = np.asarray(pe_feat, dtype=float)
            if pe.ndim == 1 and pe.shape[0] == N:
                pe = pe[:, None]
            if pe.ndim == 2 and pe.shape[0] == N:
                x = np.concatenate([x, pe], axis=1)
            else:
                print(f"[INFO] {fp}: ignore pe_feat shape {pe.shape} (expected N×D).")

        x_t = torch.tensor(x, dtype=torch.float32)

        # --- edges first ---
        edge_index = to_edge_index_from_adj(np.asarray(adj))
        E = edge_index.shape[1]

        # --- edge_attr (build BEFORE creating Data, but DO NOT ASSIGN to d yet) ---
        edge_feature = rec.get("edge_feature", None)
        edge_attr_t = build_edge_attr_from_dict(edge_feature, edge_index)

        # --- labels and noisy ---
        y_vec = rec.get("y", None)
        noisy_vec = rec.get("noise_y", None)
        M = M_DEFAULT if M_DEFAULT is not None else (
            len(np.asarray(y_vec).flatten()) if y_vec is not None
            else len(np.asarray(noisy_vec).flatten()) if noisy_vec is not None
            else 1
        )

        y_t = torch.tensor(y_vec, dtype=torch.float32) if y_vec is not None else None
        noisy_t = torch.tensor(noisy_vec, dtype=torch.float32) if noisy_vec is not None else None

        # --- masks ---
        measured_idx = rec.get("node_idx", None)         # [M]
        shortest_path = rec.get("shortest_path", None)   # [N, N]
        lc_masks = build_lightcone_masks(shortest_path, measured_idx, N, M).bool()

        # --- meta ---
        circ_idx = rec.get("circ_idx", None)
        trotter = rec.get("trotter_step", None)

        # --- NOW create the Data object ---
        d = Data(
            x=x_t,
            edge_index=edge_index,
            lightcone_masks=lc_masks,
        )
        if y_t is not None:
            d.y = y_t
        if noisy_t is not None:
            d.noisy_z = noisy_t
        if edge_attr_t is not None:
            d.edge_attr = edge_attr_t

        # attach metadata
        if circ_idx is not None:
            d.circ_idx = torch.tensor([int(circ_idx)], dtype=torch.long)
        if trotter is not None:
            d.step = torch.tensor([float(trotter)], dtype=torch.float32)
        d.num_nodes = torch.tensor([int(N)], dtype=torch.long)
        d.num_measured = torch.tensor([int(M)], dtype=torch.long)

        # --- quick sanity ---
        if hasattr(d, "edge_attr"):
            assert d.edge_attr.size(0) == d.edge_index.size(1), "edge_attr must match number of edges"
        assert d.lightcone_masks.size(0) == d.x.size(0), "mask rows must equal num_nodes"
        assert d.lightcone_masks.size(1) == int(d.num_measured.item()), "mask cols must equal M"

        graphs.append(d)

    print(f"Converted {len(graphs)} graphs.")
    return graphs


In [10]:
DATA_DIR = "/home/macula/SMATousi/Desktop/all-6-trotter-incoherent-dataset/"                 # where your data_*.pkl live
PATTERN  = "data_*.pkl"
OUT_PT   = "/home/macula/SMATousi/Desktop/graphs_from_newds.pt"  # output path

# If you know how many measured qubits (M). If unknown, we will infer from y/noise_y length.
M_DEFAULT = None  # or set to 5 if you know it's always 5


records = load_pkls(DATA_DIR, PATTERN)
graphs = convert_records_to_graphs(records)
torch.save(graphs, OUT_PT)
print("Saved:", Path(OUT_PT).resolve())

Loaded 1200 items.


  0%|                                                                                                         | 0/1200 [00:00<?, ?it/s]


IndexError: index 1 is out of bounds for axis 1 with size 1

In [6]:
records[0]

('/home/macula/SMATousi/Desktop/all-6-trotter-incoherent-dataset/data_0.pkl',
 {'x': array([[ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
           0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
           0.        ,  0.        ,  0.        ,  1.        ,  0.        ,
           0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
         [ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
           0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
           0.        ,  0.        ,  0.        ,  0.        ,  1.        ,
           0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
         [ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
           0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
           0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
           1.        ,  0.        ,  0.        ,  0.        ,  0.        ],
         [ 1. 